# Real-data examples: old vs new trended-feature denominator

Purpose of this notebook: pull ~1M real tradelines per bureau (first 10 mapped
chunks each) and show, on concrete examples, how the effective month range
(the denominator of every `percent_<rate>_<window>_months` feature) differs
between the shipping code and the proposed fix.

## The changes this notebook demonstrates

**Asset changes (model-engine, per-bureau FE2 trade.json):**

- Added `missing_data_chars` to each bureau's `PaymentPatternsAggregatorV2`
  params -- the bureau-defined code for "no rating observed this month":
  - Equifax `["*"]` -- "Rate/Status was not available for that month"
    (STS TotalView Programming Guide, p. 3-30)
  - Experian `["-"]` -- "No update received" (CIS Cross Reference Guide,
    Appendix T "Payment Profile Indicators", Segment 357.B4.5, p. 139)
  - TransUnion `["X"]` -- no data received from the subscriber / account in
    dispute (TU4.1 User Guide, Appendix C, pp. 839-840)
- Removed `placeholder: "-"` from the Experian asset: the dash is a real
  month, not formatting, so stripping it shifted every older month one
  position more recent -- misaligning lookback windows and `months_since_*`
  features. The dash now stays in the string and is excluded from denominators
  via `missing_data_chars` instead.
- Removed `placeholder: "/"` from the TransUnion asset: the TU pattern
  character set (`1-5, E, X, J, K, H, G, L, Y`; TU4.1 Appendix C,
  pp. 838-840) has no formatting characters -- the `/` was copied from the
  Equifax asset and never occurs in TU data (verified on these chunks below).
  Only Equifax keeps its placeholder, where `/` genuinely appears as a
  separator after every 12 months of history.

**Code changes (feature-engine-parts, `payment_pattern_aggregator.py`):**

- `exclude_trailing` renamed to `missing_data_chars` and made a constructor
  param fed from the asset (it was hardcoded `["*"]` in `transform`).
- `_get_effective_month_range` gained an `all_month_range` flag: `True`
  subtracts only trailing missing-data runs (legacy semantic, kept for
  `payment_history_length` = account tenure); `False` subtracts every
  occurrence anywhere in the window (spec-correct for trended denominators).
- `_construct_trended_features` anchors the denominator on
  `trimmed.str.len()` instead of the nominal window, so bureaus whose strings
  run shorter than the window aren't inflated by positions that don't exist.
- `_get_count` now wraps values in `re.escape` -- `Series.str.count` compiles
  its pattern as a regex, and Equifax's `*` would raise "nothing to repeat".

**In this notebook** the TU and Experian patterns are combined BY HAND
(combine -> trim -> add fillers, skipping `_remove_placeholder`) because the
installed assets still declare placeholders that are NOT actually placeholders
for those bureaus -- Experian's `-` is a real month and TU's `/` never occurs.
Only Equifax goes through the normal path, since its `/` really is formatting.
This means the Experian patterns here KEEP their dashes, matching the
post-placeholder-removal behavior.

**Note on the input files:** the parquet chunks under
`payment_processing_research_data/<bureau>/test/mapped/` already had the
asset's `mapping` block applied -- MapperV2 ran the StringConverter /
DateConverter / NumericConverter steps from each bureau's FE2 trade asset when
`map_and_save_mapped_data.ipynb` wrote them. So `rptDate`, `date_of_request`,
and the payment-pattern columns are already in their converted form here, and
we do NOT re-run MapperV2: `prep_bureau` below only adds the DateDiff column
and combines the pattern columns into `zest_payment_pattern`.

Denominator definitions being compared (same math as
`analyze_difference_in_denominator.ipynb`):

    eff_old = month_range          - count('#')
    eff_new = len(trimmed_pattern) - count('#') - count(missing_data_char)


In [1]:
import re
from pathlib import Path

import pandas as pd
import numpy as np

from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2
from feature_engine_parts.fe_parts_V2.preprocessors.date_diff import DateDiffV2

from configs import EQUIFAX, EXPERIAN, TRANSUNION, mapped_dir
from helpers import (
    MISSING_DATA_CHARS,
    PROJECT_COLS,
    MONTH_RANGES,
    load_asset_json,
    get_aggregator_params,
)

SPLIT = 'test'

BUREAU_CFGS = {'equifax': EQUIFAX, 'experian': EXPERIAN, 'transunion': TRANSUNION}

Y2K_DATE_OPEN

In [27]:
for b in PROJECT_COLS:
  if 'openDate' not in PROJECT_COLS[b]:
      PROJECT_COLS[b] = PROJECT_COLS[b] + ['openDate']
print({b: cols for b, cols in PROJECT_COLS.items()})

{'equifax': ['ZEST_KEY', 'date_of_request', 'rptDate', 'RATE_STATUS_CODE', 'PAYMENT_HISTORY_1_24', 'PAYMENT_HISTORY_25_36', 'PAYMENT_HISTORY_37_48', 'openDate'], 'experian': ['ZEST_KEY', 'date_of_request', 'rptDate', 'PAYMENT_PROFILE', 'openDate'], 'transunion': ['ZEST_KEY', 'date_of_request', 'rptDate', 'ppt_status', 'PAYMENT_PATTERN', 'openDate']}


In [28]:
PROJECT_COLS

{'equifax': ['ZEST_KEY',
  'date_of_request',
  'rptDate',
  'RATE_STATUS_CODE',
  'PAYMENT_HISTORY_1_24',
  'PAYMENT_HISTORY_25_36',
  'PAYMENT_HISTORY_37_48',
  'openDate'],
 'experian': ['ZEST_KEY',
  'date_of_request',
  'rptDate',
  'PAYMENT_PROFILE',
  'openDate'],
 'transunion': ['ZEST_KEY',
  'date_of_request',
  'rptDate',
  'ppt_status',
  'PAYMENT_PATTERN',
  'openDate']}

In [29]:
def prep_bureau(bureau, trade_df_mapped):
  """Returns to_use_for_payment_processing with `zest_payment_pattern` populated.

  Input must already be MAPPED (the asset's `mapping` block applied) -- the
  mapped/ chunks are, so there is no MapperV2 step here.

  Builds BOTH patterns for every bureau, plus their string lengths:
    zest_payment_pattern / zest_payment_pattern_len -- NEW behavior
      (placeholder stripped only for equifax, where '/' really is formatting)
    old_payment_pattern  / old_payment_pattern_len  -- OLD shipping behavior
      (placeholder stripped for every bureau, forced to the old asset values)
  For equifax old == new by construction. Lengths include the '#' fillers.
  """
  OLD_PLACEHOLDER = {'equifax': '/', 'experian': '-', 'transunion': '/'}

  asset = load_asset_json(bureau)
  cols = [c for c in PROJECT_COLS[bureau] if c in trade_df_mapped.columns]
  to_use_for_payment_processing = trade_df_mapped[cols].copy()

  # DateDiffV2: (date_of_request - <date>) / 30.436875 days -> months since
  date_diff = DateDiffV2(feature='rptDate', reference_feature='date_of_request',
                         new_feature='months_since_rptDate')
  to_use_for_payment_processing = date_diff.transform(to_use_for_payment_processing)
  date_diff_open = DateDiffV2(feature='openDate', reference_feature='date_of_request',
                              new_feature='months_since_openDate')
  to_use_for_payment_processing = date_diff_open.transform(to_use_for_payment_processing)

  agg = PaymentPatternsAggregatorV2(**get_aggregator_params(asset))

  # shared first step for both variants
  combined = agg._combine_payment_patterns(to_use_for_payment_processing)

  # NEW: placeholder removal ONLY for equifax. For experian/TU there is no
  # formatting character to strip:
  #   - experian: '-' is a real month, status 'No update received'
  #     (CIS Cross Reference Guide, Appendix T, Segment 357.B4.5, p. 139).
  #     Stripping it shifts every older month one position more recent; it
  #     stays in the string and is excluded from the denominators via
  #     missing_data_chars instead.
  #   - transunion: the pattern character set (1-5, E, X, J, K, H, G, L, Y;
  #     TU4.1 User Guide, Appendix C, pp. 838-840) has no formatting chars;
  #     the asset's '/' was copied from equifax and never occurs in TU data.
  #   - equifax: '/' really is formatting (a separator after every 12
  #     months of history), so it IS stripped in the new behavior too.
  if bureau == 'equifax':
      saved = agg.placeholder
      agg.placeholder = OLD_PLACEHOLDER['equifax']
      new_ppt = agg._remove_placeholder(combined)
      agg.placeholder = saved
  else:
      new_ppt = combined
  new_ppt = agg._trim(new_ppt)
  new_ppt = agg._add_fillers(to_use_for_payment_processing, new_ppt)
  to_use_for_payment_processing['zest_payment_pattern'] = new_ppt
  to_use_for_payment_processing['zest_payment_pattern_len'] = new_ppt.str.len()

  # OLD: identical steps WITH the placeholder removal for EVERY bureau,
  # forcing the old asset's placeholder so this reproduces shipping
  # behavior even when the installed (branch) asset no longer declares one.
  saved = agg.placeholder
  agg.placeholder = OLD_PLACEHOLDER[bureau]
  old_ppt = agg._remove_placeholder(combined)
  agg.placeholder = saved
  old_ppt = agg._trim(old_ppt)
  old_ppt = agg._add_fillers(to_use_for_payment_processing, old_ppt)
  to_use_for_payment_processing['old_payment_pattern'] = old_ppt
  to_use_for_payment_processing['old_payment_pattern_len'] = old_ppt.str.len()

  return to_use_for_payment_processing

In [3]:
# Load the first N_PARTS mapped chunks per bureau (~100k rows each -> ~1M
# rows per bureau). Paths come from configs.mapped_dir, not hardcoded.
N_PARTS = 10

mapped = {}
for bureau, cfg in BUREAU_CFGS.items():
    d = Path(mapped_dir(cfg, SPLIT))
    parts = sorted(d.glob('part-*.parquet'))[:N_PARTS]
    mapped[bureau] = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    print(f'[{bureau}]  {len(mapped[bureau]):,} rows from {len(parts)} chunks in {d}')

[equifax]  1,000,000 rows from 10 chunks in /home/jag/payment-processor-research/payment_processing_research_data/equifax/test/mapped
[experian]  1,000,000 rows from 10 chunks in /home/jag/payment-processor-research/payment_processing_research_data/experian/test/mapped
[transunion]  1,000,000 rows from 10 chunks in /home/jag/payment-processor-research/payment_processing_research_data/transunion/test/mapped


In [30]:
# Transform all three bureaus: PROJECT_COLS -> DateDiff -> zest_payment_pattern.
prepped = {}
for bureau, df in mapped.items():
    prepped[bureau] = prep_bureau(bureau, df)
    sample = prepped[bureau]['zest_payment_pattern'].dropna()
    print(f'[{bureau}]  {len(prepped[bureau]):,} rows; '
          f'example pattern: {sample.iloc[0][:48] if len(sample) else "(none)"}')

[equifax]  1,000,000 rows; example pattern: #11111111111111*********************************
[experian]  1,000,000 rows; example pattern: ############B00000000000000000000000000000000000
[transunion]  1,000,000 rows; example pattern: #11111111111111111111111111111111111111111111111


In [31]:
prepped_number = {'experian': prepped['experian']}

In [32]:
prepped_number

{'experian':                                                  ZEST_KEY date_of_request  \
 0       e9db23c5ad8a562636a6154edb8e3351-74f37bb0717db...      2019-12-31   
 1       2c7581e6f163431853d56d9e1503c82e-3ba7d79425a4d...      2019-12-31   
 2       34301db3d92bdcced4509b7522e31565-7075804006d46...      2019-12-31   
 3       43f13e3e5ffab4ce4871af27d7bacb5d-99e00f83fae0e...      2019-12-31   
 4       84ec57a902154d209e229258aaa81739-b3497e18feb40...      2019-12-31   
 ...                                                   ...             ...   
 999995  74ae226fb50da88d91a0df82732ce931-8fac549ea6454...      2019-12-31   
 999996  b6955e8dd620c9107a81456ca8019bbd-bde28260c6e9f...      2019-12-31   
 999997  648546b31de644bc3aed8a85d3ce8414-32f4c59619cc0...      2019-12-31   
 999998  dc0c762c6ee7dbd4e58560948b54edc9-dc6b64a144a6f...      2019-12-31   
 999999  c60dda2339e9daef293d79e8b02069d2-d1b5eef820eb4...      2019-12-31   
 
           rptDate                                

## Examples where the denominator changes

For each bureau: compute `eff_old` / `eff_new` for one illustrative window and
show a handful of real tradelines where they disagree, alongside the
`percent_DQ30+` value each method would produce. These are tradelines with
missing-data codes inside the window (or a pattern string shorter than the
window), i.e. exactly the cases the fix targets.

In [33]:
import numpy as np

In [60]:
SHOW_M     = 24   # window to illustrate
N_EXAMPLES = 8    # rows to show per bureau

# the raw bureau pattern columns, per bureau
RAW_PPT_COLS = {
  'equifax':    ['RATE_STATUS_CODE', 'PAYMENT_HISTORY_1_24',
                 'PAYMENT_HISTORY_25_36', 'PAYMENT_HISTORY_37_48'],
  'experian':   ['PAYMENT_PROFILE'],
  'transunion': ['ppt_status', 'PAYMENT_PATTERN'],
}

examples = {}
for bureau, df in prepped.items():
    missing_char = MISSING_DATA_CHARS[bureau]
    dq30_codes   = get_aggregator_params(load_asset_json(bureau))['payment_patterns']['rate']['DQ30+']
    
    ppt     = df['zest_payment_pattern'].fillna('')
    trimmed = ppt.str[:SHOW_M]
    
    # OLD: nominal window minus '#' fillers.
    # NEW: observed string length minus '#' and the bureau's missing-data char.
    eff_old = SHOW_M             - trimmed.str.count('#')
    eff_new = trimmed.str.len()  - trimmed.str.count('#') - trimmed.str.count(re.escape(missing_char))
    
    n_dq30 = trimmed.str.count('|'.join(re.escape(c) for c in dq30_codes))
    
    out = df.copy()
    out[f'trimmed_{SHOW_M}'] = trimmed
    out['eff_old']           = eff_old
    out['eff_new']           = eff_new
    out['n_DQ30+']           = n_dq30
    out['pct_DQ30+_old']     = (n_dq30 / eff_old).round(4)
    out['pct_DQ30+_new']     = (n_dq30 / eff_new.where(eff_new > 0)).round(4)
    examples[bureau] = out
    out['diff'] = np.abs(out['pct_DQ30+_old'] - out['pct_DQ30+_new'])
    
    changed = out[out['eff_old'] != out['eff_new']].sort_values(by='diff', ascending=False)
    print(f"\n[{bureau}]  missing_char={missing_char!r}  window={SHOW_M}m  "
          f"{len(changed):,}/{len(out):,} rows change "
          f"({len(changed) / len(out) * 100:.1f}%)")
    
    # slim display: raw pattern cols + trimmed window + eff/pct comparison only
    show_cols = ([c for c in RAW_PPT_COLS[bureau] if c in changed.columns]
                 + [f'trimmed_{SHOW_M}', 'eff_old', 'eff_new',
                    'pct_DQ30+_old', 'pct_DQ30+_new', 'diff'])
    with pd.option_context('display.max_colwidth', None, 'display.max_columns', None):
        display(changed.head(N_EXAMPLES)[show_cols])


[equifax]  missing_char='*'  window=24m  281,257/1,000,000 rows change (28.1%)


,RATE_STATUS_CODE,PAYMENT_HISTORY_1_24,PAYMENT_HISTORY_25_36,PAYMENT_HISTORY_37_48,trimmed_24,eff_old,eff_new,pct_DQ30+_old,pct_DQ30+_new,diff
618685,Z,************/************,/************,/************,#Z**********************,23,1,0.0435,1.0,0.9565
657832,6,************/************,/************,/************,#6**********************,23,1,0.0435,1.0,0.9565
658000,6,************/************,/************,/************,#6**********************,23,1,0.0435,1.0,0.9565
657967,9,************/************,/************,/************,#9**********************,23,1,0.0435,1.0,0.9565
657928,9,************/************,/************,/************,#9**********************,23,1,0.0435,1.0,0.9565
657916,6,************/************,/************,/************,#6**********************,23,1,0.0435,1.0,0.9565
657911,6,************/************,/************,/************,#6**********************,23,1,0.0435,1.0,0.9565
657885,6,************/************,/************,/************,#6**********************,23,1,0.0435,1.0,0.9565



[experian]  missing_char='-'  window=24m  278,721/1,000,000 rows change (27.9%)


,PAYMENT_PROFILE,trimmed_24,eff_old,eff_new,pct_DQ30+_old,pct_DQ30+_new,diff
933341,G,#G,23,1,0.0435,1.0,0.9565
461270,G,#G,23,1,0.0435,1.0,0.9565
579422,G,#G,23,1,0.0435,1.0,0.9565
121205,G,#G,23,1,0.0435,1.0,0.9565
412597,G,#G,23,1,0.0435,1.0,0.9565
512173,G,#G,23,1,0.0435,1.0,0.9565
549106,G,#G,23,1,0.0435,1.0,0.9565
50880,G,#G,23,1,0.0435,1.0,0.9565



[transunion]  missing_char='X'  window=24m  257,902/1,000,000 rows change (25.8%)


,ppt_status,PAYMENT_PATTERN,trimmed_24,eff_old,eff_new,pct_DQ30+_old,pct_DQ30+_new,diff
505557,L,None,#L,23,1,0.0435,1.0,0.9565
229667,L,None,#L,23,1,0.0435,1.0,0.9565
229963,G,None,#G,23,1,0.0435,1.0,0.9565
832932,L,None,#L,23,1,0.0435,1.0,0.9565
229813,L,None,#L,23,1,0.0435,1.0,0.9565
978147,G,None,#G,23,1,0.0435,1.0,0.9565
229738,G,None,#G,23,1,0.0435,1.0,0.9565
833133,L,None,#L,23,1,0.0435,1.0,0.9565


# Just Experian: testing whether the `-` removal was deleting real history

The payment pattern is anchored at the **report date**: after dropping the
`#` fillers (which only represent the gap between the report date and the
pull date), the pattern's length = how many months of history it covers,
counting back from the report date.

We can check that against ground truth: the account's age at the report date
(`months_open_at_report` = open date → report date, via DateDiffV2). A
calendar-true pattern should cover the account's whole life.

One adjustment: the pipeline trims every pattern to **48 characters**, so a
pattern can never cover more than 48 months no matter how old the account
is. That's why we cap the age at 48 before comparing — otherwise every
account older than 48 months would look like it has "missing" history when
it's really just the trim.

The comparison:

  age_minus_new_pattern = min(age_at_report, 48) - len(new pattern, no '#')
  age_minus_old_pattern = min(age_at_report, 48) - len(old pattern, no '#')

- **0** — the pattern covers the account's whole (visible) life
- **> 0** — that many months of the account's life are missing from the pattern

If removing the `-` was harmless formatting, old and new should look the
same. If the `-` was real history, the OLD pattern should fall short of the
account's age by exactly the number of dashes stripped — and the NEW pattern
should sit at ~0.

In [62]:
experian_df = examples['experian']

In [ ]:
experian_df['zest_payment_pattern'].str.endswith('X', na=False).sum()

In [63]:
changed = experian_df[experian_df['zest_payment_pattern']!=experian_df['old_payment_pattern']]

In [64]:
changed['zest_payment_pattern_no_fill'] = changed['zest_payment_pattern'].str.replace('#', '', regex=False)
changed['old_payment_pattern_no_fill']  = changed['old_payment_pattern'].str.replace('#', '', regex=False)
changed['zest_pattern_len_no_fill'] = changed['zest_payment_pattern_no_fill'].str.len()
changed['old_pattern_len_no_fill']  = changed['old_payment_pattern_no_fill'].str.len()

# 2) months between open date and report date = account age AS OF the report
#    date (same 30.436875-day month as the other DateDiffs)
date_diff_open_rpt = DateDiffV2(feature='openDate', reference_feature='rptDate',
                              new_feature='months_open_at_report')
changed = date_diff_open_rpt.transform(changed)

In [65]:
changed['age_minus_new_pattern'] = (changed['months_open_at_report'].clip(upper=48)
                                  - changed['zest_pattern_len_no_fill'])
changed['age_minus_old_pattern'] = (changed['months_open_at_report'].clip(upper=48)
                                  - changed['old_pattern_len_no_fill'])

In [66]:
print(changed[['age_minus_new_pattern', 'age_minus_old_pattern']].describe(percentiles = (.1, .50,.75,.90,.95,.99)).round(2))

       age_minus_new_pattern  age_minus_old_pattern
count              151261.00              151261.00
mean                    1.87                   7.70
std                     6.77                  11.51
min                   -47.90                 -46.49
10%                    -0.61                   0.00
50%                     0.00                   1.31
75%                     0.00                  11.60
90%                     4.76                  26.00
95%                    16.59                  35.00
99%                    36.00                  44.00
max                    45.00                  47.00


In [69]:
changed['months_since_open_date_clipped']= changed['months_open_at_report'].clip(upper=48)

In [72]:
changed[changed['age_minus_new_pattern']>=4][['zest_payment_pattern_no_fill', 'old_payment_pattern_no_fill', 'zest_pattern_len_no_fill', 'old_pattern_len_no_fill', 'rptDate', 'openDate','months_since_open_date_clipped', 'date_of_request']] 

,zest_payment_pattern_no_fill,old_payment_pattern_no_fill,zest_pattern_len_no_fill,old_pattern_len_no_fill,rptDate,openDate,months_since_open_date_clipped,date_of_request
172,L-LL-LLL-L-LLLLLL-L-L-LLL-LLLLL-LL-L-L,LLLLLLLLLLLLLLLLLLLLLLLLLLL,38.0,27.0,2017-07-20,2012-06-05,48.000000,2019-12-31
211,B-----C,BC,7.0,2.0,2013-01-21,2008-04-17,48.000000,2019-12-31
214,CCCCCCCCCCCC1111C22222--221111,CCCCCCCCCCCC1111C22222221111,30.0,28.0,2019-12-11,2012-05-21,48.000000,2019-12-31
360,B--C,BC,4.0,2.0,2015-02-28,2013-02-18,24.312614,2019-12-31
427,B-----GG,BGG,8.0,3.0,2019-12-01,2013-12-01,48.000000,2019-12-31
...,...,...,...,...,...,...,...,...
999674,C-CCCC,CCCCC,6.0,5.0,2019-11-30,2008-03-25,48.000000,2019-12-31
999693,96---66666-666654322C,9666666666654322C,21.0,17.0,2015-06-17,2005-06-03,48.000000,2019-12-31
999822,BCCCCCCCCCCCCCCCC-C,BCCCCCCCCCCCCCCCCC,19.0,18.0,2014-05-26,2011-12-07,29.602251,2019-12-31
999826,B00--CCCCC---C-----C,B00CCCCCCC,20.0,10.0,2012-10-31,2008-02-06,48.000000,2019-12-31


In [74]:
changed[changed['age_minus_new_pattern']>=4][['months_since_rptDate']].describe()

,months_since_rptDate
count,15867.000000
mean,36.765945
std,37.488171
min,0.131420
25%,1.018501
50%,25.462535
75%,65.282655
max,171.995318


In [76]:
changed[changed['age_minus_new_pattern']>=4][['old_pattern_len_no_fill', 'months_since_open_date_clipped','age_minus_old_pattern']].describe()

,old_pattern_len_no_fill,months_since_open_date_clipped,age_minus_old_pattern
count,15867.000000,15867.000000,15867.000000
mean,16.686267,41.831734,25.145467
std,11.085541,9.945529,11.835032
min,2.000000,7.260929,5.000000
25%,7.000000,37.783117,15.000000
50%,14.000000,48.000000,24.000000
75%,25.000000,48.000000,35.042927
max,43.000000,48.000000,46.000000


In [77]:
changed[changed['age_minus_new_pattern']<=4][['old_pattern_len_no_fill', 'months_since_open_date_clipped','age_minus_old_pattern']].describe()

,old_pattern_len_no_fill,months_since_open_date_clipped,age_minus_old_pattern
count,135799.000000,135799.000000,135799.000000
mean,35.856744,41.553133,5.696389
std,13.942843,11.531384,9.613741
min,1.000000,-8.148011,-46.488675
25%,26.000000,38.900183,0.000000
50%,41.000000,48.000000,0.784698
75%,48.000000,48.000000,7.343231
max,48.000000,48.000000,47.000000


# Pattern coverage vs account age — old vs new (changed experian tradelines)

`age_minus_*_pattern` = account age at the report date (capped at the
48-month trim) minus the pattern's length with the `#` fillers removed —
i.e. how many months of the account's visible life the pattern FAILS to
cover (0 = calendar-true).

- **The new pattern matches the account's real age**: 0.0 at the median AND
the 75th percentile — for most affected tradelines, keeping the dashes
makes the pattern span the account's entire visible life exactly.
- **The old pattern was systematically too short**: short by 1.3 months at
the median, 11.6 at the 75th percentile, and 26+ months for the worst
decile — the dash-stripping was erasing real account history, not
formatting, and every rating older than a stripped dash was reported as
that many months too recent.
- **The new pattern's residual gaps are honest**: where it still falls short
(95th pct = 16.6), the bureau simply never reported those early months at
all — the fix recovers every month the bureau marked "no update received"
but cannot invent history the bureau never sent. (Small negatives are
fractional-month rounding; the deep-negative extremes are a handful of
open-date data quirks present in both columns.)

In [79]:
transunion_df = examples['transunion']

In [80]:
changed_tu = transunion_df[transunion_df['zest_payment_pattern']!=transunion_df['old_payment_pattern']]

In [82]:
len(changed_tu)

0

## TransUnion: old vs new patterns are identical (0 changed rows)

Expected. The only TU change was removing `placeholder: "/"` from the asset,
and `/` never occurs in TransUnion data — the TU4.1 pattern character set
(`1-5, E, X, J, K, H, G, L, Y`; Appendix C, pp. 838-840) has no formatting
characters; the `/` was copied over from the equifax asset, where it really
does appear as an every-12-months separator. With nothing to strip,
`_remove_placeholder` was always a no-op for TU, so the old and new patterns
are bit-identical across all rows. This doubles as a control: it confirms
the placeholder removal changes experian only.


In [83]:
transunion_df['zest_payment_pattern'].str.endswith('X', na=False).sum()

54478

In [84]:
for bureau, ch in [('transunion', 'X'), ('experian', '-')]:
  ppt = prepped[bureau]['zest_payment_pattern'].dropna()
  ends_star = ppt.str.endswith('*')   # what the OLD code looked for
  ends_ch   = ppt.str.endswith(ch)    # what the NEW code looks for
  print(f"{bureau}: {len(ppt):,} tradelines | "
        f"end with '*' (old hardcoded): {ends_star.sum():,} ({ends_star.mean():.2%}) | "
        f"end with {ch!r} (bureau char): {ends_ch.sum():,} ({ends_ch.mean():.2%})")

# payment_history_length (the trailing-only effective range), before vs after.
#   OLD: len - count('#') - trailing run of '*'   (always 0 for these bureaus)
#   NEW: len - count('#') - trailing run of the bureau's char
# trailing-run length = len(s) - len(s with that char stripped from the right)
for bureau, ch in [('transunion', 'X'), ('experian', '-')]:
  ppt  = prepped[bureau]['zest_payment_pattern'].fillna('')
  base = ppt.str.len() - ppt.str.count('#')

  old_eff = base - (ppt.str.len() - ppt.str.rstrip('*').str.len())
  new_eff = base - (ppt.str.len() - ppt.str.rstrip(ch).str.len())

  months_removed = old_eff - new_eff
  changed = months_removed > 0
  print(f"\n{bureau}: payment_history_length changes for "
        f"{changed.sum():,}/{len(ppt):,} tradelines ({changed.mean():.2%})")
  if changed.any():
      print("months removed where changed:")
      print(months_removed[changed].describe().round(2).to_string())

transunion: 1,000,000 tradelines | end with '*' (old hardcoded): 0 (0.00%) | end with 'X' (bureau char): 54,478 (5.45%)
experian: 998,690 tradelines | end with '*' (old hardcoded): 0 (0.00%) | end with '-' (bureau char): 22,815 (2.28%)

transunion: payment_history_length changes for 54,478/1,000,000 tradelines (5.45%)
months removed where changed:
count    54478.00
mean        10.38
std         12.75
min          1.00
25%          1.00
50%          3.00
75%         18.00
max         48.00

experian: payment_history_length changes for 22,815/1,000,000 tradelines (2.28%)
months removed where changed:
count    22815.00
mean        16.01
std         14.09
min          1.00
25%          4.00
50%         11.00
75%         24.00
max         47.00
